# CMSC 320 Final Project — Checkpoint 2
## S&P 500 Stock Market EDA

This notebook preprocesses the S&P 500 dataset and explores it with three statistical analyses: descriptive statistics on daily returns, a one-way ANOVA across sectors, and a t-test comparing high vs low volume days.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import f_oneway, ttest_ind

I load three CSV files: daily stock prices, company metadata, and the aggregate index.

In [ ]:
stocks    = pd.read_csv('../data/archive/sp500_stocks.csv')
companies = pd.read_csv('../data/archive/sp500_companies.csv')
index     = pd.read_csv('../data/archive/sp500_index.csv')

print(stocks.shape, companies.shape, index.shape)
stocks.head()

Next I parse dates, coerce numeric columns, merge sector labels, and compute daily returns.

In [ ]:
stocks['Date'] = pd.to_datetime(stocks['Date'])
for col in ['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']:
    stocks[col] = pd.to_numeric(stocks[col], errors='coerce')

stocks.columns    = stocks.columns.str.lower().str.replace(' ', '_')
companies.columns = companies.columns.str.lower().str.replace(' ', '_')

stocks = stocks.sort_values(['symbol', 'date']).reset_index(drop=True)
stocks = stocks.merge(companies[['symbol', 'sector']], on='symbol', how='left')
stocks['daily_return'] = stocks.groupby('symbol')['close'].pct_change()

df = stocks.dropna(subset=['daily_return', 'sector']).copy()
print(df.shape)
df.head()

Here is a quick overview of the clean dataset.

In [ ]:
print('Rows:', df.shape[0], ' Columns:', df.shape[1])
print('Unique stocks:', df['symbol'].nunique())
print('Unique sectors:', df['sector'].nunique())
print('Date range:', df['date'].min().date(), 'to', df['date'].max().date())

## Analysis 1: Distribution of Daily Returns

I use descriptive statistics and a histogram to summarize the shape and spread of daily returns across all S&P 500 stocks.

In [ ]:
returns = df['daily_return']
print(returns.describe())

plt.figure(figsize=(8, 4))
plt.hist(returns.clip(-0.1, 0.1), bins=80, color='steelblue', edgecolor='white')
plt.xlabel('Daily Return')
plt.ylabel('Count')
plt.title('Distribution of Daily Returns')
plt.show()

The mean daily return is close to zero, and the distribution is roughly bell-shaped, but there are still many extreme positive and negative return days. This tells me that daily returns are not perfectly normal.

## Analysis 2: Do Sectors Have Different Mean Returns? (ANOVA)

I run a one-way ANOVA to test whether mean daily returns differ significantly across the 11 S&P 500 GICS sectors.

In [ ]:
groups = [g['daily_return'].values for _, g in df.groupby('sector')]
stat, p = f_oneway(*groups)
print('F-statistic:', round(stat, 4))
print('p-value:', p)

plt.figure(figsize=(10, 5))
sns.boxplot(data=df, x='sector', y='daily_return', showfliers=False)
plt.xticks(rotation=45, ha='right')
plt.xlabel('Sector')
plt.ylabel('Daily Return')
plt.title(f'Daily Returns by Sector  (ANOVA p = {p:.2e})')
plt.tight_layout()
plt.show()

Since the p-value is much smaller than 0.05, I reject the null hypothesis that all sectors have the same mean daily return. This suggests that average daily returns differ across sectors.

## Analysis 3: High Volume vs Low Volume Days (t-test)

I split trading days at the median volume and use a two-sample t-test to check whether high-volume days tend to produce larger absolute returns than low-volume days.

In [ ]:
median_vol = df['volume'].median()
high = df[df['volume'] >= median_vol]['daily_return'].abs()
low  = df[df['volume'] <  median_vol]['daily_return'].abs()

stat, p = ttest_ind(high, low, equal_var=False)
print('p-value:', p)
print('high-vol mean:', round(high.mean(), 6))
print('low-vol mean: ', round(low.mean(), 6))

plt.figure(figsize=(6, 5))
plt.boxplot([low, high], labels=['Low Volume', 'High Volume'], showfliers=False)
plt.ylabel('|Daily Return|')
plt.title('Absolute Return: High vs Low Volume Days')
plt.show()

Since the p-value is much smaller than 0.05, I reject the null hypothesis that high-volume and low-volume days have the same average return magnitude. High-volume days tend to have larger price swings.